In [96]:
import datetime
import numpy as np
import pandas as pd

kaggle = pd.read_csv("./building-data-genome-project-2/data/meters/processed/df_summary.csv", index_col=0)
metadata = pd.read_csv("./building-data-genome-project-2/data/metadata/metadata.csv")
weather = pd.read_csv("./building-data-genome-project-2/data/weather/weather.csv")

sites = {"Lamb":"Gunnar", "Mouse":"Korv", "Robin":"Trapano", "Shrew":"Vorr", "Wolf":"Nymeria"}
old_site_ids = metadata["site_id_kaggle"].unique()
site_ids = {1: 143, 5: 52, 12:13}

RANDOM_SEED = 345


#### Encode meters

In [137]:
# Encode meters
np.random.seed(RANDOM_SEED)

meter_df = pd.DataFrame(data={"meter":kaggle["meter"].unique(), "meter_code": [np.random.randint(0, 1076) for m in kaggle["meter"].unique()]})
meter_df.to_csv("./kaggle_data/meters.csv")

np.random.seed(RANDOM_SEED)
d = {m: np.random.randint(0, 1076) for m in kaggle["meter"].unique()}
kaggle["meter"].replace(d, inplace=True)
kaggle.head()

#### Data preprocessing

In [98]:

kaggle = kaggle.loc[kaggle["building_id"].isin(metadata.loc[metadata["site_id"].isin(sites.keys())].loc[metadata["building_id"].isna()==False]["building_id"]).values]
weather = weather.loc[weather["site_id"].isin(sites.keys())]
metadata = metadata.loc[metadata["site_id"].isin(sites.keys())].loc[metadata["building_id"].isna()==False]
# kaggle["timestamp"] = pd.to_datetime(kaggle["timestamp"], format="%Y-%m-%d %H:%M:%S") 
# kaggle["day"] = kaggle["timestamp"].apply(lambda s: s.date())

In [100]:
kaggle = kaggle[["building_id", "day", "meter", "meter_reading"]].groupby(["building_id", "day", "meter"], as_index=False).sum()
# kaggle.to_csv("kaggle_europe.csv")
# metadata.to_csv("metadata_europe.csv")

In [102]:
old_ids = kaggle["building_id"].unique()
np.random.seed(seed=RANDOM_SEED)
new_ids = np.random.randint(0, 1000, 2*len(old_ids))
new_ids = list(set(new_ids))[:len(old_ids)]

In [103]:
kaggle["building_id"].replace({b:i for b,i in zip(old_ids, new_ids)}, inplace=True)
metadata["building_id"].replace({b:i for b,i in zip(old_ids, new_ids)}, inplace=True)

In [112]:
metadata["site_id"].replace(sites, inplace=True)
metadata["site_id_kaggle"].replace(site_ids, inplace=True)
weather["site_id"].replace(sites, inplace=True)

In [ ]:
metadata.drop(columns=["building_id_kaggle"], inplace=True)


#### Adjust values

In [101]:
DAY_OFFSET = 7
weather["timestamp"] = pd.to_datetime(weather["timestamp"], format="%Y-%m-%d %H:%M:%S")
kaggle["day"]= pd.to_datetime(kaggle["day"], format="%Y-%m-%d")
kaggle["day"] = kaggle["day"].apply(lambda s: s+datetime.timedelta(days=DAY_OFFSET))
weather["timestamp"] = weather["timestamp"].apply(lambda s: s+datetime.timedelta(days=DAY_OFFSET))


In [126]:
np.random.seed(RANDOM_SEED)
adjust_values = np.random.rand(*weather.shape[::-1])

for i, col in enumerate(weather.columns):
    if col not in ["timestamp", "site_id"]:
        weather[col] += adjust_values[i].round(1)

In [131]:
np.random.seed(RANDOM_SEED)
adjust_values = np.random.rand(kaggle.shape[0])
kaggle["meter_reading"] += adjust_values.round(1)

#### Write data

In [149]:
kaggle.to_csv("./kaggle_data/full_kaggle.csv")
metadata.to_csv("./kaggle_data/metadata.csv")
weather.to_csv("./kaggle_data/weather.csv")

#### Split

In [ ]:
start = datetime.datetime.strptime("20160201", "%Y%m%d")
test =  datetime.datetime.strptime("20171031","%Y%m%d")
end = datetime.datetime.strptime("20171231","%Y%m%d")

In [ ]:
train = kaggle.loc[kaggle["day"]<test].loc[kaggle["day"]>start]
test = kaggle.loc[kaggle["day"]<end].loc[kaggle["day"]>test]
weather = weather.loc[weather["timestamp"]<end].loc[weather["timestamp"]>start]

In [168]:
weather.to_csv("./kaggle_data/train.csv")
weather.to_csv("./kaggle_data/weather.csv")

In [175]:
test["meter_reading"] = test["meter_reading"].apply(lambda s: round(s, 2))
test.head()

,building_id,day,meter,meter_reading
1326,2,2017-11-01,595,0.20
1327,2,2017-11-01,207,162.71
1328,2,2017-11-02,595,0.30
1329,2,2017-11-02,207,186.63
1330,2,2017-11-03,595,0.90


In [182]:
test.drop(columns=["meter_reading"]).reset_index(drop=True).to_csv("./kaggle_data/test.csv")
test["meter_reading"].reset_index(drop=True).to_csv("./kaggle_data/solution.csv")

In [185]:
train.reset_index(drop=True).to_csv("./kaggle_data/train.csv")